# Deploy Integrated Supervisor Agent to Unity Catalog

This notebook packages the `agent_integrated.py` code and registers it as a Model in Unity Catalog.

## What Gets Deployed

The **Integrated Supervisor Agent** combines:
- **MCP Tools** for data engineering questions (pipeline health, data quality, lineage, anomalies)
- **Genie Space** for business analytics questions (revenue, sales, customers, KPIs)
- **Intelligent Routing** to automatically direct questions to the appropriate backend

## Authentication

**Service Principal OAuth** (used for both MCP and Genie):
- Credentials loaded from `intellipipe` secrets scope:
  - `databricks_host`
  - `sp_client_id` 
  - `sp_client_secret`
- OAuth token obtained via M2M flow
- Same token used for both MCP server and Genie Space API

## Configuration

**From model_config** (passed to serving endpoint):
- `WORKSPACE_URL` - Databricks workspace URL
- `GENIE_SPACE_ID` - Genie Space ID (default: 01f133e2518a1c609256e4485119b82d)
- `LLM_ENDPOINT` - Foundation model endpoint name
- `MCP_CONNECTION_NAME` - UC connection name pointing to MCP server

In [0]:
# %pip install --upgrade 'mlflow>=2.16.0' databricks-sdk databricks-mcp
# dbutils.library.restartPython()

In [0]:
# pip_requirements=[
#     "mlflow>=2.16.0",
#     "databricks-langchain>=0.1.0",
#     "langchain-core",
#     "databricks-sdk>=0.20.0",
#     "requests>=2.31.0"
# ],

In [0]:
import os
import mlflow
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec


# ── Configuration ──────────────────────────────────────────────────────────
UC_CATALOG = "capstone_project"
UC_SCHEMA = "capstone_schema"
MODEL_NAME = "supervisor_mcp_agent"
REGISTERED_MODEL = f"{UC_CATALOG}.{UC_SCHEMA}.{MODEL_NAME}"

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
MCP_CONNECTION_NAME = "intellipipe_mcp_connection"

# Set MLflow to use Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Ensure you have an experiment set, or it will use the default notebook experiment
# mlflow.set_experiment("/Shared/Intellipipe_Supervisor_Agent")

In [0]:
# ── Define Signature & Sample Input ────────────────────────────────────────
input_schema = Schema([ColSpec("string", "messages")])
output_schema = Schema([ColSpec("string", "content")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

sample_input = {
    "messages": [
        {"role": "user", "content": "What is the health status of the DLT pipeline?"}
    ]
}

In [0]:
# ── Log and Register the Integrated Agent ──────────────────────────────────────
# Install dependencies
import subprocess
import sys
import pandas as pd
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksUCConnection

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "databricks-langchain", "langchain-core"])

print(f"🚀 Logging Integrated Supervisor Agent to {REGISTERED_MODEL}...")
print("\n📦 Authentication:")
print("   • Service Principal credentials loaded from 'intellipipe' secrets scope")
print("   • OAuth token used for both MCP tools and Genie Space")
print("   • GENIE_SPACE_ID from model_config (default: 01f133e2518a1c609256e4485119b82d)")

# Import IntegratedSupervisorAgent from local file
import importlib.util
import importlib

# Load the agent module dynamically
agent_file_path = "/Workspace/Users/capstoneproject196@gmail.com/Intellipipe-capstone/Intellipipe/supervisor-mcp/agent_integrated.py"
spec = importlib.util.spec_from_file_location("agent_integrated", agent_file_path)
agent_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(agent_module)

# Get the IntegratedSupervisorAgent class
IntegratedSupervisorAgent = agent_module.IntegratedSupervisorAgent

# Runtime config (passed to load_context via model_config)
model_config = {
    "WORKSPACE_URL": "https://dbc-5e2e6f3e-0d80.cloud.databricks.com",
    "GENIE_SPACE_ID": "01f133e2518a1c609256e4485119b82d", # Can be overridden
    "LLM_ENDPOINT": LLM_ENDPOINT,
    "MCP_CONNECTION_NAME": MCP_CONNECTION_NAME,
}

# Sample inputs for validation
sample_input = pd.DataFrame({
    "messages": [
        str([{"role": "user", "content": "What is the health status of the DLT pipeline?"}]),
        str([{"role": "user", "content": "Show me top 5 customers by revenue"}]),
    ]
})

with mlflow.start_run(run_name="integrated_supervisor_deployment"):
    
    model_info = mlflow.pyfunc.log_model(
        artifact_path="supervisor_agent",
        python_model=IntegratedSupervisorAgent(),
        code_paths=["agent_integrated.py"],
        signature=signature,
        input_example=sample_input,
        model_config=model_config,
        
        pip_requirements=[
            "mlflow>=2.16.0",
            "databricks-langchain>=0.1.0",
            "langchain-core",
            "databricks-sdk>=0.20.0",
            "requests>=2.31.0"
        ],
        
        # Declare resources for auto-permission provisioning
        resources=[
            DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
            DatabricksUCConnection(connection_name=MCP_CONNECTION_NAME),
        ],
        
        registered_model_name=REGISTERED_MODEL,
    )

print(f"\n✅ Successfully registered model! Version: {model_info.registered_model_version}")
print("\n🎯 NEXT STEPS:")
print("1. Go to the 'Serving' UI in Databricks")
print("2. Edit the 'supervisor_mcp' endpoint (or create new)")
print(f"3. Update to use {REGISTERED_MODEL} version {model_info.registered_model_version}")
print("4. The endpoint will automatically get permissions for LLM and MCP connection")
print("\n🔐 Required Secrets (in 'intellipipe' scope):")
print("   • databricks_host")
print("   • sp_client_id")
print("   • sp_client_secret")
print("\n⚡ The agent uses these to obtain OAuth tokens for MCP and Genie")